## variable selection looped over a range of rensonance-masses

### *RSGrav reco-analysis*

### *–––– Set up ––––*

In [1]:
import numpy as np
import awkward as ak
import uproot
import matplotlib.pyplot as plt
import hist
import hist.dask as hda
import dask
import coffea.processor as processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import vector
import json
import pandas as pd

NanoAODSchema.warn_missing_crossrefs = False

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/fcc.py:5: FutureWarning: In version 2025.1.0 (target date: 2024-12-31 11:59:59-06:00), this will be an error.
To raise these warnings as errors (and get stack traces to find out where they're called), run
    import warnings
    warnings.filterwarnings("error", module="coffea.*")
after the first `import coffea` or use `@pytest.mark.filterwarnings("error:::coffea.*")` in pytest.
Issue: coffea.nanoevents.methods.vector will be removed and replaced with scikit-hep vector. Nanoevents schemas internal to coffea will be migrated. Otherwise please consider using that package!.
  from coffea.nanoevents.methods import vector


In [2]:
with open("samples.json", 'r') as f:
    fileset = json.load(f)
    
bulk_samples = [name for name in fileset if name.startswith("BulkGravToWW")]
rs_samples = [name for name in fileset if name.startswith("RSGravToWW")]
all_samples  = bulk_samples + rs_samples

### *––– Helper Functions –––*

In [3]:
def extract_semi_subjet_pairs(fatjets, all_subjets, category="Semi-Leptonic"):
    indices, pairs, n_skipped = [], [], 0
    for i, (fj_list, sj_list) in enumerate(zip(fatjets, all_subjets)):
        if len(fj_list) == 0:
            continue
        fj   = fj_list[0]
        # always cast to int: works for both Bulk and RS
        raw1 = fj_list[0].subJetIdx1
        raw2 = fj_list[0].subJetIdx2
        idx1, idx2 = int(raw1), int(raw2)
        if idx1<0 or idx2<0 or idx1>=len(sj_list) or idx2>=len(sj_list):
            n_skipped += 1
            continue
        try:
            s1, s2 = sj_list[idx1], sj_list[idx2]
            v1 = vector.obj(pt=s1["pt"], eta=s1["eta"], phi=s1["phi"], mass=s1["mass"])
            v2 = vector.obj(pt=s2["pt"], eta=s2["eta"], phi=s2["phi"], mass=s2["mass"])
            indices.append(i)
            pairs .append((v1, v2))
        except Exception:
            n_skipped += 1

    print(f"[{category}] Valid pairs: {len(pairs)}, Skipped: {n_skipped}")
    return np.array(indices, dtype=int), pairs


def extract_had_subjet_pairs(fatjets, category="Hadronic"):
    indices, pairs, n_skipped = [], [], 0
    for i, fj in enumerate(fatjets):
        sjs = fj.subjets
        if len(sjs) < 2:
            n_skipped += 1
            continue
        try:
            s1, s2 = sjs[0], sjs[1]
            v1 = vector.obj(pt=s1["pt"], eta=s1["eta"], phi=s1["phi"], mass=s1["mass"])
            v2 = vector.obj(pt=s2["pt"], eta=s2["eta"], phi=s2["phi"], mass=s2["mass"])
            indices.append(i)
            pairs .append((v1, v2))
        except Exception:
            n_skipped += 1

    print(f"[{category}] Valid pairs: {len(pairs)}, Skipped: {n_skipped}")
    return np.array(indices, dtype=int), pairs

In [4]:
# -------------------------
# ECFs Helpers
# -------------------------
def delta_phi(phi1, phi2):
    dphi = phi1 - phi2
    return (dphi + np.pi) % (2*np.pi) - np.pi

def delta_r(eta1, phi1, eta2, phi2):
    return np.sqrt((eta1 - eta2)**2 + delta_phi(phi1, phi2)**2)

def _ecf2(parts, beta=1):
    """
    parts: awkward array [events, constituents] with fields pt, eta, phi
    returns: e2^{(beta)} per jet
    """
    pt_sum = ak.sum(parts.pt, axis=-1)
    comb2 = ak.combinations(parts, 2, fields=['a','b'])
    dr    = delta_r(comb2.a.eta, comb2.a.phi, comb2.b.eta, comb2.b.phi)
    num   = ak.sum(comb2.a.pt * comb2.b.pt * (dr**beta), axis=-1)
    return num / (pt_sum**2)

def _ecf3(parts, beta=1):
    """
    e3^{(beta)}. Will be 0 (or None) if fewer than 3 constituents.
    """
    pt_sum = ak.sum(parts.pt, axis=-1)
    comb3  = ak.combinations(parts, 3, fields=['a','b','c'])
    # pairwise ΔR products
    dr_ab = delta_r(comb3.a.eta, comb3.a.phi, comb3.b.eta, comb3.b.phi)
    dr_ac = delta_r(comb3.a.eta, comb3.a.phi, comb3.c.eta, comb3.c.phi)
    dr_bc = delta_r(comb3.b.eta, comb3.b.phi, comb3.c.eta, comb3.c.phi)
    prod_dr = (dr_ab * dr_ac * dr_bc)**beta
    num = ak.sum(comb3.a.pt * comb3.b.pt * comb3.c.pt * prod_dr, axis=-1)
    return num / (pt_sum**3)

def compute_ecfs(jets, use_constituents=True, beta=1):
    """
    jets: awkward array of AK8 jets
    If use_constituents is False, we'll use jets.subjets instead.
    Returns e2, e3, (optionally) C2 and D2 masks.
    """
    # pick constituents or fallback to subjets
    if use_constituents and hasattr(jets, "constituents"):
        parts = jets.constituents
    else:
        parts = jets.subjets

    # e2 (β=1) and e2 (β=2)
    e2_b1 = _ecf2(parts, beta=beta)
    e2_b2 = _ecf2(parts, beta=2)

    # e3 and ratios
    have3  = ak.num(parts) >= 3
    e3_raw = _ecf3(parts, beta=beta)
    e3     = ak.where(have3, e3_raw, 0.0)
    C2     = ak.where((e2_b1 > 0) & have3, e3 / (e2_b1**2), 0.0)
    D2     = ak.where((e2_b1 > 0) & have3, e3 / (e2_b1**3), 0.0)

    # pTD = sqrt(sum(pt^2)) / sum(pt)
    # two‑subjet kinematics:
    pt_sum  = ak.sum(parts.pt, axis=-1)
    pt2_sum = ak.sum(parts.pt**2, axis=-1)
    pTD     = ak.where(pt_sum > 0, (pt2_sum**0.5) / pt_sum, 0.0)

    have2 = ak.num(parts) >= 2
    pt1   = ak.where(have2, parts.pt[:, 0], 0.0)
    pt2   = ak.where(have2, parts.pt[:, 1], 0.0)

    # momentum sharing z = min(pt1,pt2)/(pt1+pt2)
    z = ak.where( have2,ak.where(pt1 < pt2, pt1, pt2) / pt_sum, 0.0)

    # ΔR between the two leading subjets
    dr12 = ak.where( have2, delta_r(parts.eta[:, 0], parts.phi[:, 0],
    parts.eta[:, 1], parts.phi[:, 1]), 0.0)
    return e2_b1, e2_b2, e3, C2, D2, pTD, z, dr12

In [5]:
def compute_cos_theta_star(idx_list, subjet_pairs, category_name=""):
    cos_theta_star = []
    idx_success = []
    n_success = 0
    n_skipped = 0

    for j, (vec1, vec2) in enumerate(subjet_pairs):
        w_lab = vec1 + vec2
        if w_lab.mass < 1e-3 or w_lab.E <= 1e-6:
            n_skipped += 1
            continue

        beta3 = -w_lab.to_beta3()
        if beta3.mag < 1e-6:
            n_skipped += 1
            continue

        # rapidity‐sort
        q = vec1 if vec1.rapidity < vec2.rapidity else vec2
        q_rest = q.boost_beta3(beta3)
        if q_rest.to_beta3().mag < 1e-6:
            n_skipped += 1
            continue

        w_hat = w_lab.to_beta3().unit()
        q_hat = q_rest.to_beta3().unit()
        cos_theta_star.append(q_hat.dot(w_hat))
        n_success += 1
        idx_success.append(j)

    if n_success > 0:
        arr = np.array(cos_theta_star)
        print(f"[{category_name}] Computed cosθ* for {n_success} pairs, Skipped: {n_skipped}")
        print(f"[{category_name}] range: {arr.min():.3f} → {arr.max():.3f}")
        print(f"[{category_name}] counts (neg/zero/pos): "
              f"{np.sum(arr<0)}/{np.sum(arr==0)}/{np.sum(arr>0)}")
    else:
        print(f"[{category_name}] All {n_skipped} pairs skipped; no cosθ*.")

    return np.array(idx_success, dtype=int), np.array(cos_theta_star)

### *–––main–––*

In [8]:
for sample_name in bulk_samples:

    # ––– STEP 0: LOAD EVENTS ––––
    events = NanoEventsFactory.from_root(
        fileset[sample_name]["files"],
        entry_stop=10000,
        metadata=fileset[sample_name]["metadata"],
        schemaclass=NanoAODSchema,
        delayed=False,
    ).events()

    # ––– STEP 1: TIGHT LEPTON SELECTION  –––
    muons = events.Muon
    electrons = events.Electron
    
    muons_tight = muons[
        (muons.pt > 35) & (muons.eta < 2.4) & (muons.tightId) &
        (((muons.pt < 10) & (muons.dxy < 0.01)) | ((muons.pt >= 10) & (muons.dxy < 0.02))) &
        (muons.dz < 0.05) 
    # & (muons.pfIsoId >= 4)
    ]
    
    electrons_tight = electrons[
        (electrons.pt > 30) & (electrons.eta < 2.4) & (((electrons.pt < 15) & 
        (electrons.dxy < 0.01)) | ((electrons.pt >= 15) & 
        (electrons.dxy < 0.02))) & (electrons.dxy < 0.05) & # dxy tighter (electrons.dz < 0.05) &
        (electrons.pfRelIso03_all < 0.15) & (electrons.lostHits <= 1) & (electrons.convVeto == True)
    ]
    
    tight_muon_count = ak.num(muons_tight)
    tight_electron_count = ak.num(electrons_tight)
    loose_muons = muons[(muons.isPFcand) & (muons.mediumId) & (muons.pfRelIso03_all < 0.4)]
    loose_muons = loose_muons[loose_muons.pt > 10]
    loose_electrons = electrons[electrons.cutBased >= 2]
    loose_electrons = loose_electrons[loose_electrons.pt > 10]
    loose_lepton_count = ak.num(loose_muons) + ak.num(loose_electrons)
    
    single_lepton_mask = ((tight_muon_count + tight_electron_count) == 1) & (loose_lepton_count == 1)

    # Masks for exactly one tight lepton
    one_tight_electron_mask = (tight_electron_count == 1) & (tight_muon_count == 0) & (loose_lepton_count == 1)
    one_tight_muon_mask = (tight_muon_count == 1) & (tight_electron_count == 0) & (loose_lepton_count == 1)
    
    # Apply masks to original tight collections
    tight_electron_one = electrons_tight[one_tight_electron_mask]
    tight_muon_one = muons_tight[one_tight_muon_mask]

    # ––– STEP 2: AK8 CLEANING –––
    # FatJets cuts
    clean_fatJets = events.FatJet[(events.FatJet.pt > 150) & (events.FatJet.eta < 2.4) & (events.FatJet.msoftdrop > 0)]
    #Jets cuts
    clean_Jets = events.Jet[(events.Jet.pt > 30) & (events.Jet.eta < 4.6)]

    #Removing AK4(Jet) jets overlapping with AK8(FatJets) jets
    jets_fatjets = ak.cartesian({"x": clean_Jets, "y": clean_fatJets})
    jets_iso_f = ((jets_fatjets["x"].eta-jets_fatjets["y"].eta)**2+(jets_fatjets["x"].phi-jets_fatjets["y"].phi)**2>0.8**2)
    jets_fatjets = jets_fatjets[jets_iso_f]
    jets, fj = ak.unzip(jets_fatjets)

    # Require events with at least 1 clean AK8 FatJets
    AK8jets_candidates_mask = ak.num(clean_fatJets) >= 1
    Wjets_candidates = clean_fatJets[AK8jets_candidates_mask]
    leading_W_jet = Wjets_candidates[:, 0]
    leading_W_jet_pt = leading_W_jet.pt

    # Require events with at least  clean AK8 FatJets
    AK8jets_2_mask = ak.num(clean_fatJets) >= 2
    Wjets_2_candidates = clean_fatJets[AK8jets_2_mask]
    second_W_jet = Wjets_2_candidates[:, 1]
    second_W_jet_pt = second_W_jet.pt

    # ––– STEP 3: EVENT CATEGORIZATION –––
    #tight lepton masks
    n_tight_muons = ak.num(muons_tight)
    n_tight_electrons = ak.num(electrons_tight)
    
    # event masks
    semi_leptonic_mask = (
        ((n_tight_muons == 1) & (n_tight_electrons == 0)) |
        ((n_tight_muons == 0) & (n_tight_electrons == 1))) & (ak.num(clean_fatJets) >= 1)
    
    fully_hadronic_mask = (
        (n_tight_muons == 0) & (n_tight_electrons == 0) &
        (ak.num(clean_fatJets) >= 2))

    # semi-leptonic W decay: 
    semi_leptonic_fatjets = clean_fatJets[semi_leptonic_mask]
    all_semi_subjets       = events.SubJet[semi_leptonic_mask]
    # fully hadronic W decay: 
    fully_hadronic_fatjets = clean_fatJets[fully_hadronic_mask]

    # For plotting (flatten to leading/subleading jets)
    semi_leptonic_leading_jet = semi_leptonic_fatjets[:, 0]
    fully_hadronic_leading_jet = fully_hadronic_fatjets[:, 0]
    fully_hadronic_second_jet = fully_hadronic_fatjets[:, 1]
    
    # — STEP 4: Subjet pairs + cosθ* — 
    print(f"[DEBUG] semi_leptonic_fatjets.len = {len(semi_leptonic_fatjets)}")
    print(f"[DEBUG] all_semi_subjets.len      = {len(all_semi_subjets)}")
    
    idx_evt_semi, pairs_semi = extract_semi_subjet_pairs(semi_leptonic_fatjets,events.SubJet[semi_leptonic_mask],category="Semi‑Leptonic")
    idx_evt_h1,  pairs_h1  = extract_had_subjet_pairs(fully_hadronic_leading_jet,category="Hadronic Lead")
    idx_evt_h2,  pairs_h2  = extract_had_subjet_pairs(fully_hadronic_second_jet,category="Hadronic Sublead")
    
    # compute cosθ* and get back event‐level indices in one shot
    idx_final_semi, cos_semi = compute_cos_theta_star(idx_evt_semi, pairs_semi,   "Semi‑Leptonic")
    idx_final_h1,   cos_h1   = compute_cos_theta_star(idx_evt_h1,   pairs_h1,     "Hadronic Lead")
    idx_final_h2,   cos_h2   = compute_cos_theta_star(idx_evt_h2,   pairs_h2,     "Hadronic Sublead")

    # ————————————— STEP 4.5: Compute additional 2‑subjet features —————————————
    # for each channel, run compute_ecfs on the full leading‐jet arrays
    USE_CONSTITUENTS = False
    e2b1_semi, e2b2_semi, e3_semi, C2_semi, D2_semi, pTD_semi, z_semi, dr12_semi = \
        compute_ecfs(semi_leptonic_leading_jet, USE_CONSTITUENTS, beta=1)

    e2b1_h1, e2b2_h1, e3_h1, C2_h1, D2_h1, pTD_h1, z_h1, dr12_h1 = \
        compute_ecfs(fully_hadronic_leading_jet, USE_CONSTITUENTS, beta=1)

    e2b1_h2, e2b2_h2, e3_h2, C2_h2, D2_h2, pTD_h2, z_h2, dr12_h2 = \
        compute_ecfs(fully_hadronic_second_jet, USE_CONSTITUENTS, beta=1)

    # helper to pick out only the events that survived cosθ*
    def select_np(arr, idx):
        # fill any None with 0.0, then index and convert to numpy
        return ak.to_numpy(ak.fill_none(arr[idx], 0.0))

    # apply the three idx_final_* masks
    e2b1_semi = select_np(e2b1_semi, idx_final_semi)
    e2b2_semi = select_np(e2b2_semi, idx_final_semi)
    e3_semi   = select_np(e3_semi,   idx_final_semi)
    C2_semi   = select_np(C2_semi,   idx_final_semi)
    D2_semi   = select_np(D2_semi,   idx_final_semi)
    pTD_semi  = select_np(pTD_semi,  idx_final_semi)
    z_semi    = select_np(z_semi,    idx_final_semi)
    dr12_semi = select_np(dr12_semi, idx_final_semi)

    e2b1_h1 = select_np(e2b1_h1, idx_final_h1)
    e2b2_h1 = select_np(e2b2_h1, idx_final_h1)
    e3_h1   = select_np(e3_h1,   idx_final_h1)
    C2_h1   = select_np(C2_h1,   idx_final_h1)
    D2_h1   = select_np(D2_h1,   idx_final_h1)
    pTD_h1  = select_np(pTD_h1,  idx_final_h1)
    z_h1    = select_np(z_h1,    idx_final_h1)
    dr12_h1 = select_np(dr12_h1, idx_final_h1)

    e2b1_h2 = select_np(e2b1_h2, idx_final_h2)
    e2b2_h2 = select_np(e2b2_h2, idx_final_h2)
    e3_h2   = select_np(e3_h2,   idx_final_h2)
    C2_h2   = select_np(C2_h2,   idx_final_h2)
    D2_h2   = select_np(D2_h2,   idx_final_h2)
    pTD_h2  = select_np(pTD_h2,  idx_final_h2)
    z_h2    = select_np(z_h2,    idx_final_h2)
    dr12_h2 = select_np(dr12_h2, idx_final_h2)

    # ——— STEP 5: Select the jets that survived and build DataFrames ———
    sel_semi = semi_leptonic_leading_jet[idx_final_semi]
    sel_h1   = fully_hadronic_leading_jet[idx_final_h1]
    sel_h2   = fully_hadronic_second_jet[idx_final_h2]
    
    # semi-lep DF
    df_semi = pd.DataFrame({
        "pt":    ak.to_numpy(sel_semi.pt),
        "eta":   ak.to_numpy(sel_semi.eta),
        "msoft": ak.to_numpy(sel_semi.msoftdrop),
        "cos":   cos_semi,
        "e2_b1":  e2b1_semi,
        "e2_b2":  e2b2_semi,
        "e3":     e3_semi,
        "C2":     C2_semi,
        "D2":     D2_semi,
        "pTD":    pTD_semi,
        "z":      z_semi,
        "dr12":   dr12_semi,
    })
    df_semi["channel"], df_semi["label"] = "semi", 0
    df_semi.to_hdf(f"new_h5_files_raw/{sample_name}_semi_features.h5", key="df", mode="w")
    
    # hadronic-lead DF
    df_h1 = pd.DataFrame({
        "pt":    ak.to_numpy(sel_h1.pt),
        "eta":   ak.to_numpy(sel_h1.eta),
        "msoft": ak.to_numpy(sel_h1.msoftdrop),
        "cos":   cos_h1,
        "e2_b1":  e2b1_h1,
        "e2_b2":  e2b2_h1,
        "e3":     e3_h1,
        "C2":     C2_h1,
        "D2":     D2_h1,
        "pTD":    pTD_h1,
        "z":      z_h1,
        "dr12":   dr12_h1,
    })
    df_h1["channel"], df_h1["label"] = "had1", 0
    df_h1.to_hdf(f"new_h5_files_raw/{sample_name}_had1_features.h5", key="df", mode="w")
    
    # hadronic-sublead DF
    df_h2 = pd.DataFrame({
        "pt":    ak.to_numpy(sel_h2.pt),
        "eta":   ak.to_numpy(sel_h2.eta),
        "msoft": ak.to_numpy(sel_h2.msoftdrop),
        "cos":   cos_h2,
        "e2_b1":  e2b1_h2,
        "e2_b2":  e2b2_h2,
        "e3":     e3_h2,
        "C2":     C2_h2,
        "D2":     D2_h2,
        "pTD":    pTD_h2,
        "z":      z_h2,
        "dr12":   dr12_h2,
    })
    df_h2["channel"], df_h2["label"] = "had2", 0
    df_h2.to_hdf(f"new_h5_files_raw/{sample_name}_had2_features.h5", key="df", mode="w")
    
    print(f"{sample_name}: semi→{len(df_semi)}, had1→{len(df_h1)}, had2→{len(df_h2)}")

[DEBUG] semi_leptonic_fatjets.len = 3643
[DEBUG] all_semi_subjets.len      = 3643
[Semi‑Leptonic] Valid pairs: 3163, Skipped: 480
[Hadronic Lead] Valid pairs: 4857, Skipped: 16
[Hadronic Sublead] Valid pairs: 4779, Skipped: 94
[Semi‑Leptonic] Computed cosθ* for 3163 pairs, Skipped: 0
[Semi‑Leptonic] range: -0.867 → 0.855
[Semi‑Leptonic] counts (neg/zero/pos): 1576/0/1587
[Hadronic Lead] Computed cosθ* for 4857 pairs, Skipped: 0
[Hadronic Lead] range: -0.909 → 0.873
[Hadronic Lead] counts (neg/zero/pos): 2402/0/2455
[Hadronic Sublead] Computed cosθ* for 4779 pairs, Skipped: 0
[Hadronic Sublead] range: -0.878 → 0.931
[Hadronic Sublead] counts (neg/zero/pos): 2415/0/2364


/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:251: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


BulkGravToWW_1000: semi→3163, had1→4857, had2→4779
[DEBUG] semi_leptonic_fatjets.len = 3813
[DEBUG] all_semi_subjets.len      = 3813
[Semi‑Leptonic] Valid pairs: 3360, Skipped: 453
[Hadronic Lead] Valid pairs: 5098, Skipped: 23
[Hadronic Sublead] Valid pairs: 4967, Skipped: 154
[Semi‑Leptonic] Computed cosθ* for 3360 pairs, Skipped: 0
[Semi‑Leptonic] range: -0.971 → 0.874
[Semi‑Leptonic] counts (neg/zero/pos): 1695/0/1665
[Hadronic Lead] Computed cosθ* for 5098 pairs, Skipped: 0
[Hadronic Lead] range: -0.953 → 0.949
[Hadronic Lead] counts (neg/zero/pos): 2546/0/2552
[Hadronic Sublead] Computed cosθ* for 4967 pairs, Skipped: 0
[Hadronic Sublead] range: -0.980 → 0.978
[Hadronic Sublead] counts (neg/zero/pos): 2500/0/2467


/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:251: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


BulkGravToWW_2000: semi→3360, had1→5098, had2→4967
